# 03 — Portfólio da noite: o que 24 h realmente compram

## Por que isto não é um treino de 24 h

Não existe "treinar por 24 h" nesta configuração. Os 34 runs locais pararam por early
stopping entre a época **12 e 33** de 40 — mediana 23,5, nenhum bateu o teto. Numa A100
cada um cai para minutos: um monólito de 24 h simplesmente termina.

Pior: a curva de validação é **plana e ruidosa a partir da época ~4**. No `ft_last12_s43`
o melhor valor sai na época 4 e depois oscila ±15–30% por mais oito épocas. A escolha do
melhor checkpoint é em parte loteria, e parte do ±0,32 entre sementes é ruído de *seleção
de época*, não de otimização. Isso se combate com replicação, não com run mais longo.

## Os arms, em ordem de esperança fundamentada

| arm | hipótese | mecanismo |
|---|---|---|
| **A. geometria solar** | `concat` no lugar de `image_only` | `ImageOnlyModel.forward` diz no próprio docstring que **ignora** `features`. Os 38 runs anteriores descartaram as 9 features — inclusive `solar_elevation` — e o viés residual é monotônico em elevação. Único arm com mecanismo para esse defeito. |
| **B. capacidade** | ViT-B/14 | maior passo da tabela do DINOv2 (ver notebook 02) |
| **C. resolução** | 322/448 px | interpola em direção à grade nativa 37×37 |
| **D. ensemble** | 5 sementes | ataca **variância, não viés** — não espere que conserte o MBE |

## Segurança operacional

O timeout por inatividade do Colab **só conta quando a execução termina**. Como cada run
para na época ~20, a VM fica elegível para reciclagem antes de você buscar os artefatos.
Por isso **todo arquivamento acontece na mesma célula do treino**.


## 1. Runtime e GPU

**Antes de rodar:** `Runtime > Change runtime type > A100 GPU`, com *High-RAM* **desligado**
— o pico medido é 482 MiB no ViT-S e ~4 GB no ViT-B, e a variante de 80 GB custa +39% de
unidades por memória que não usamos.

Custo: A100-40GB ≈ 5,4 unidades/h, então 24 h ≈ 130 CU ≈ **26% da cota mensal do Pro+**
(500 CU). Confira a taxa real em *View resources*, no menu superior direito.


In [ ]:
import subprocess

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True, check=False).stdout)

## 2. Ambiente

O pacote exige CPython >= 3.14, que o Colab normalmente não traz — por isso o `uv`
provisiona um interpretador próprio. **A instalação do torch CUDA é obrigatória e
verificada:** o extra `allsky` fixa uma wheel de CPU, e um torch de CPU aqui invalida
a sessão inteira.

Esta é a única célula que não pode vir do `_colab_runner`: é ela que clona o repo onde
o runner mora.


In [ ]:
import os
import subprocess
import sys

REPO = "https://github.com/Bruno-Mascarenhas/micrometeorology.git"
BRANCH = "main"
WORKDIR = "/content/micrometeorology"

if not os.path.exists(WORKDIR):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO, WORKDIR], check=True)
subprocess.run(["pip", "install", "-q", "uv"], check=True)
subprocess.run(["uv", "python", "install", "3.14"], cwd=WORKDIR, check=True)
subprocess.run(["uv", "venv", "--python", "3.14", ".venv"], cwd=WORKDIR, check=True)
subprocess.run(["uv", "sync", "--locked", "--extra", "allsky"], cwd=WORKDIR, check=True)
subprocess.run(
    [
        "uv",
        "pip",
        "install",
        "--python",
        ".venv/bin/python",
        "--reinstall",
        "--torch-backend",
        "cu130",
        "torch==2.13.0",
    ],
    cwd=WORKDIR,
    check=True,
)

PY = f"{WORKDIR}/.venv/bin/python"
os.environ["PATH"] = f"{WORKDIR}/.venv/bin:" + os.environ["PATH"]
sys.path.insert(0, f"{WORKDIR}/notebooks/colab")

verify = subprocess.run(
    [PY, "-c", "import torch; print(torch.__version__, torch.cuda.is_available())"],
    capture_output=True,
    text=True,
    check=False,
)
print(verify.stdout)
if "True" not in verify.stdout:
    raise RuntimeError("torch sem CUDA — pare e reinstale antes de treinar")

## 3. Dados e artefatos

`stage_bundle` copia o bundle para o SSD local, desempacota e roda `validate-dataset`.
Os três passos importam: treinar de `/content/drive` é FUSE e a leitura fria domina a
época, e um bundle truncado treinaria em silêncio sem a validação.

`ARTIFACTS` no Drive é o que sobrevive à sessão — o timeout por inatividade do Colab só
conta **quando a execução termina**, e todo run para por early stopping na época ~20.


In [ ]:
import os

import _colab_runner as runner
from google.colab import drive

BUNDLE = "/content/drive/MyDrive/labmim/allsky-mm/bundle.tar.gz"
DATA = "/content/allsky-mm"
ARTIFACTS = "/content/drive/MyDrive/labmim/runs/allsky-mm"

drive.mount("/content/drive")
os.makedirs(ARTIFACTS, exist_ok=True)
ROOT = runner.stage_bundle(BUNDLE, DATA, python=PY)

## 4. Hardware e ajustes que dependem dele

`bf16` existe em toda GPU do Colab menos a T4 (Turing). O código local roda `fp16`
porque a 2060 não tem alternativa; aqui a escolha é automática.

O probe roda no interpretador do venv, que é onde o torch com CUDA está instalado.


In [ ]:
import json

probe = subprocess.run(
    [
        PY,
        "-c",
        'import json, sys; sys.path.insert(0, "' + WORKDIR + '/notebooks/colab"); '
        "import _colab_runner as r; print(json.dumps(r.probe_accelerator()))",
    ],
    capture_output=True,
    text=True,
    check=True,
)
HW = json.loads(probe.stdout.strip().splitlines()[-1])
AMP_DTYPE = HW["amp_dtype"]
WORKERS = min(8, HW["cpus"])
print(HW)
print(f"amp={AMP_DTYPE}  workers={WORKERS}")

## 5. Arm A — geometria solar

O mais barato e o único com mecanismo. Se o perfil de MBE por elevação achatar, você
achou a causa do viés que resistiu a quatro perdas e a todas as profundidades.


In [ ]:
from pathlib import Path

CFG = Path(WORKDIR) / "configs/allsky/experiments/colab"
OUT = Path("/content/out")
rows = []

TRAIN = {
    "backbone_lr": 1e-5,
    "epochs": 40,
    "batch_size": 64,
    "num_workers": WORKERS,
    "amp": {"enabled": True, "dtype": AMP_DTYPE},
}
MODEL = {"backbone_frozen": False, "unfreeze_last_n": 12, "image_size": 224}


def arm(name, model_frag, model, train, seed, note):
    """Run one portfolio arm, archive it in the same cell, and record its row."""
    config = runner.write_config(
        CFG / f"{name}.yaml",
        extends=["../_base.yaml", f"../../models/{model_frag}.yaml"],
        name=name,
        output_dir=str(OUT / name),
        seed=seed,
        data_root=ROOT,
        model=model,
        train=train,
        targets=runner.DHI_ONLY_TARGETS,
        note=note,
    )
    row = runner.run_experiment(config)
    print(f"{name:<20} {row.get('status')} rmse={row.get('rmse')} mbe={row.get('mbe')}")
    print(" ", runner.archive(str(OUT / name), ARTIFACTS, config=config))
    row["arm"] = name.split("_")[0]
    rows.append(row)
    return row


for seed in (42, 43, 44):
    arm(
        f"geom_s{seed}",
        "concat",
        MODEL,
        TRAIN,
        seed,
        "concat consome as 9 features que image_only descarta",
    )
for seed in (42, 43, 44):
    arm(f"ctrl_s{seed}", "image_only", MODEL, TRAIN, seed, "controle pareado, mesma sessao")

runner.summarise(rows)

### O teste que importa no arm A

Não é o RMSE médio — é o **perfil de MBE por elevação solar**. Um ganho genuíno de
geometria *achata* a inclinação; um que só melhora a média não achata nada.


In [ ]:
import pandas as pd


def mbe_profile(prefix):
    """Mean MBE per solar-elevation band across every seed of one arm."""
    series = []
    for path in Path(ARTIFACTS).glob(f"{prefix}*/eval-test/stratified.csv"):
        table = pd.read_csv(path)
        table = table[
            (table.target == "dhi")
            & (table.metric == "mbe")
            & (table.stratum_kind == "solar_elevation")
        ]
        series.append(table.set_index("stratum")["value"])
    if not series:
        raise FileNotFoundError(
            f"nenhum {ARTIFACTS}/{prefix}*/eval-test/stratified.csv: "
            "sem artefato nao ha o que comparar"
        )
    print(f"{prefix}: {len(series)} semente(s)")
    return pd.concat(series, axis=1).mean(axis=1)


BANDS = ["10-20", "20-35", "35-50", "50-90"]
compare = pd.DataFrame(
    {"controle": mbe_profile("ctrl_s"), "geometria": mbe_profile("geom_s")}
).reindex(BANDS)
compare["delta"] = compare["geometria"] - compare["controle"]
print(compare.round(2).to_string())

# Sem as quatro bandas nos dois arms nao ha inclinacao para comparar: com NaN,
# `nan < 0.7 * nan` e False e o ramo `else` publicava "a geometria nao era a
# causa" — uma hipotese falseada em cima de dado ausente.
faltando = compare[compare[["controle", "geometria"]].isna().any(axis=1)].index.tolist()
if faltando:
    print(f"bandas ausentes em algum arm: {faltando} — nada a concluir")
else:
    slope_ctrl = abs(compare["controle"].iloc[-1] - compare["controle"].iloc[0])
    slope_geom = abs(compare["geometria"].iloc[-1] - compare["geometria"].iloc[0])
    print(f"inclinacao 10-20 -> 50-90:  controle {slope_ctrl:.1f}  geometria {slope_geom:.1f}")
    if slope_geom < 0.7 * slope_ctrl:
        print("ACHATOU: a geometria explicava o vies.")
    else:
        print("NAO achatou: a geometria nao era a causa. Registre como defeito em aberto.")

## 6. Arms B e C — capacidade e resolução

Rode só o que o notebook 02 indicar. Repetir aqui um arm que já perdeu lá queima cota
(A100-40GB ≈ 5,4 CU/h; 24 h ≈ 130 CU ≈ 26% do mês).


In [ ]:
BEST_BACKBONE = "dinov2_vitb14"  # ajuste conforme o notebook 02
RUN_B = True
RUN_C = True

if RUN_B:
    for seed in (42, 43):
        arm(
            f"capB_s{seed}",
            "concat",
            {**MODEL, "backbone": BEST_BACKBONE},
            {**TRAIN, "batch_size": 48},
            seed,
            "ViT-B sobre o melhor arm de features",
        )
if RUN_C:
    arm(
        "resC_s42",
        "concat",
        {**MODEL, "backbone": BEST_BACKBONE, "image_size": 448},
        {**TRAIN, "batch_size": 12},
        42,
        "448px = grade 32x32, em direcao a nativa 37x37",
    )

runner.summarise(rows)

## 7. Arm D — ensemble de sementes

Lakshminarayanan et al. usam M=5 como padrão. **Ataca variância, não viés**: se o MBE
sobreviver ao arm A, o ensemble não vai consertá-lo, e dizer o contrário seria mentira.
O ganho esperado é em RMSE e calibração.


In [ ]:
import numpy as np

BEST_ARM = "geom"  # ajuste para o arm vencedor
for seed in (45, 46):
    arm(f"{BEST_ARM}_s{seed}", "concat", MODEL, TRAIN, seed, "membro extra do ensemble")

members = [r for r in rows if r.get("status") == "ok" and r.get("arm") == BEST_ARM]
individual = np.array([r["rmse"] for r in members])
print(
    f"{len(individual)} membros: RMSE individual "
    f"{individual.mean():.2f} +- {individual.std(ddof=1):.2f}"
)

frames = [
    pd.read_parquet(p) for p in Path(ARTIFACTS).glob(f"{BEST_ARM}_s*/eval-test/predictions.parquet")
]
if len(frames) > 1:
    print("colunas disponiveis:", list(frames[0].columns))
    pred_col = next((c for c in frames[0].columns if "pred" in c and "dhi" in c), None)
    true_col = next((c for c in frames[0].columns if "target_dhi" in c), None)
    if pred_col and true_col:
        stacked = np.mean([f[pred_col].to_numpy() for f in frames], axis=0)
        observed = frames[0][true_col].to_numpy()
        residual = stacked - observed
        print(f"ensemble RMSE {np.sqrt((residual**2).mean()):.2f} MBE {residual.mean():+.2f}")

## 8. Fechamento

Grava o índice que o próximo notebook lê para saber contra o que comparar.


In [ ]:
import json

frame = runner.summarise(rows)
frame.to_csv(f"{ARTIFACTS}/portfolio.csv", index=False)
summary = {
    "hardware": HW,
    "n_runs": len(rows),
    "melhor": frame.iloc[0].to_dict() if len(frame) else None,
}
with open(f"{ARTIFACTS}/portfolio_resumo.json", "w") as handle:
    json.dump(summary, handle, indent=2, default=str)
print(frame.to_string())
print("artefatos em", ARTIFACTS)
print(
    "metrics.json = o numero | stratified.csv = onde o erro mora |",
    "predictions.parquet = ensemble e analise por amostra | metrics.csv = curva",
)